In [1]:
import pandas as pd
import numpy as numpy

In [7]:
import os
import re
from pathlib import Path

INPUT_DIR = "data"
OUTPUT_DIR = "snippets"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Define regex pattern to match EPS-related keywords
EPS_KEYWORDS = [
    r'\beps\b',
    r'earnings per share',
    r'eps guidance',
    r'adjusted eps',
    r'diluted eps',
    r'per-share earnings'
]
keyword_pattern = re.compile('|'.join(EPS_KEYWORDS), re.IGNORECASE)

def get_word_window(text, keyword_regex, window=20):
    words = text.split()
    snippets = []

    for i, word in enumerate(words):
        joined = ' '.join(words[max(0, i - window): min(len(words), i + window + 1)])
        if keyword_regex.search(word):
            snippets.append(joined)

    # Remove exact duplicates while preserving order
    seen = set()
    unique_snippets = []
    for s in snippets:
        if s not in seen:
            seen.add(s)
            unique_snippets.append(s)

    return '\n'.join(unique_snippets)

# Process each .txt file
for filepath in Path(INPUT_DIR).glob("*.txt"):
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        content = f.read()

    snippet_text = get_word_window(content, keyword_pattern)
    if snippet_text.strip():  # Save only if there's a match
        output_file = Path(OUTPUT_DIR) / filepath.name
        with open(output_file, "w", encoding="utf-8") as out:
            out.write(snippet_text)


In [18]:
import openai
import pandas as pd
from pathlib import Path
import time


client = openai.OpenAI(api_key="APIKEY")
SNIPPETS_DIR = "snippets"
MAX_FILES = 10

def extract_eps_from_text(text):
    prompt = f"""
You are a financial analyst assistant.

You will be given a text snippet from an SEC 8-K filing. Your task is to extract **only the QUARTERLY EPS guidance**, and return it in a **precise format**.

Please follow these strict instructions:

1. Only extract **EPS (Earnings Per Share)** guidance.
2. Ignore any **annual, full-year, multi-year, or daily** EPS numbers. Only quarterly EPS is valid.
3. If the text includes **both diluted and undiluted EPS**, extract only the **diluted EPS**.
4. If multiple EPS values are mentioned, extract all valid **quarterly EPS** values **separated by semicolons**.
5. You must **only return EPS values** in one of the following formats:
   - `$X.XX–$Y.YY` (for a range)
   - `$X.XX` (for a single value)
   - `None` (if no valid quarterly EPS guidance is present)

6. You must **NOT** return:
   - Any explanation, commentary, or text
   - Any alternative formats (e.g., "$1.20 +/- $0.05", "$1.20 to $1.25", bullet points, markdown)
   - Any full-year or FY values (e.g., "FY 2024: $3.31")
   - Phrases like “The EPS is…” or “EPS guidance is…”

Here is the input text:

{text}

Your answer must strictly follow the allowed format only.
"""

    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=15
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"⚠️ Error on file: {e}")
        return "ERROR"

results = []

for i, snippet_file in enumerate(sorted(Path(SNIPPETS_DIR).glob("*.txt"))):
    if i >= MAX_FILES:
        break

    with open(snippet_file, "r", encoding="utf-8") as f:
        snippet = f.read().strip()

    if snippet:
        print(f"[{i+1}/{MAX_FILES}] Processing {snippet_file.name}...")
        eps_value = extract_eps_from_text(snippet)
    else:
        eps_value = "EMPTY"

    results.append({
        "filename": snippet_file.name,
        "eps_extracted": eps_value
    })

    time.sleep(1)

df = pd.DataFrame(results)
df.to_csv("eps_extracted_results_first100.csv", index=False)
print("✅ Done! Saved to eps_extracted_results_first100.csv")

[1/10] Processing 0000002488-25-000009.txt...
[2/10] Processing 0000002969-25-000012.txt...
[3/10] Processing 0000004127-25-000009.txt...
[4/10] Processing 0000004904-25-000025.txt...
[5/10] Processing 0000004962-25-000007.txt...
[6/10] Processing 0000004962-25-000034.txt...
[7/10] Processing 0000004977-25-000006.txt...
[8/10] Processing 0000005272-25-000017.txt...
[9/10] Processing 0000005513-25-000002.txt...
[10/10] Processing 0000005513-25-000013.txt...
✅ Done! Saved to eps_extracted_results_first100.csv


In [15]:
df

,filename,eps_extracted
0,0000002488-25-000009.txt,The numeric EPS guidance mentioned in the text...
1,0000002969-25-000012.txt,The numeric EPS guidance mentioned in the text...
2,0000004127-25-000009.txt,The numeric EPS guidance mentioned in the text...
3,0000004904-25-000025.txt,The numeric EPS guidance mentioned in the text...
4,0000004962-25-000007.txt,$15.00 to $15.50
...,...,...
95,0000040987-25-000067.txt,The numeric EPS guidance mentioned in the text...
96,0000043920-25-000014.txt,$0.39
97,0000045876-25-000015.txt,The numeric EPS guidance mentioned in the text...
98,0000046080-25-000063.txt,$1.04
